In [14]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("Using device:", DEVICE)

Using device: cuda


## Load Data

In [15]:
TRAIN_PATH = "/kaggle/input/pirate-pain-dataset-feature-engineering/pirate_pain_train.csv"
TRAIN_LABELS_PATH = "/kaggle/input/pirate-pain-dataset-feature-engineering/pirate_pain_train_labels.csv"
TEST_PATH = "/kaggle/input/pirate-pain-dataset-feature-engineering/pirate_pain_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
train_labels_df = pd.read_csv(TRAIN_LABELS_PATH)
test_df = pd.read_csv(TEST_PATH)

ID_COL = "sample_index"
TIME_COL = "time"
TARGET_COL = "label"
STATIC_COL = "is_pirate"

train_df = train_df.merge(train_labels_df[[ID_COL, TARGET_COL]], on=ID_COL, how="left")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Classes:", train_labels_df[TARGET_COL].unique())

Train shape: (105760, 41)
Test shape: (211840, 40)
Classes: ['no_pain' 'low_pain' 'high_pain']


## Preprocessing: Create Static Features and Drop Columns

In [16]:
def check_pirate(row):
    return 0 if row.get("n_legs", "two") == "two" else 1

if "n_legs" in train_df.columns:
    train_df["is_pirate"] = train_df.apply(check_pirate, axis=1)
    test_df["is_pirate"] = test_df.apply(check_pirate, axis=1)
    
    for col in ["n_legs", "n_hands", "n_eyes"]:
        if col in train_df.columns:
            train_df.drop(columns=col, inplace=True)
        if col in test_df.columns:
            test_df.drop(columns=col, inplace=True)

TO_DROP = ["joint_11", "joint_30"]
if all(col in train_df.columns for col in TO_DROP):
    train_df.drop(columns=TO_DROP, inplace=True)
if all(col in test_df.columns for col in TO_DROP):
    test_df.drop(columns=TO_DROP, inplace=True)

## Outlier Removal

In [17]:
JOINT_COLS = [c for c in train_df.columns if c.startswith("joint_")]
print("Number of joint features:", len(JOINT_COLS))

def compute_global_thresholds(df, joint_cols, q=0.999, multiplier=10.0):
    thresholds = {}
    for c in joint_cols:
        q_high = df[c].quantile(q)
        thresholds[c] = float(q_high * multiplier)
    return thresholds

def correct_glitches(df, joint_cols, thresholds, id_col="sample_index", time_col="time",
                     ratio_thresh=1e4, verbose=True, labels_df=None, label_col="label",
                     exclude_labels=None):
    import math
    eps = 1e-12
    total_cells = 0
    total_segments = 0
    
    working_df = df
    if labels_df is not None:
        if label_col not in df.columns:
            working_df = df.merge(labels_df[[id_col, label_col]], on=id_col, how="left")
    
    exclude_mask = None
    if exclude_labels is not None and label_col in working_df.columns:
        exclude_mask = working_df[label_col].isin(exclude_labels)

    for sid in df[id_col].unique():
        sub_mask = df[id_col] == sid
        sub_idx_sorted = df.loc[sub_mask].sort_values(time_col).index
        times = df.loc[sub_idx_sorted, time_col].to_numpy()
        nT = len(sub_idx_sorted)
        
        if exclude_mask is not None:
            subject_exclude_mask = exclude_mask.loc[sub_idx_sorted].to_numpy()
        else:
            subject_exclude_mask = None

        for col in joint_cols:
            vals_orig = df.loc[sub_idx_sorted, col].to_numpy(dtype=float)
            vals_new = vals_orig.copy()
            cand = vals_orig > thresholds[col]
            if not np.any(cand):
                continue

            cand_idx = np.where(cand)[0]
            runs = []
            start = cand_idx[0]
            prev = cand_idx[0]
            for k in cand_idx[1:]:
                if k == prev + 1:
                    prev = k
                else:
                    runs.append((start, prev))
                    start = k
                    prev = k
            runs.append((start, prev))

            for a, b in runs:
                if subject_exclude_mask is not None:
                    if np.any(subject_exclude_mask[a:b + 1]):
                        continue
                
                neighbour_idxs = []
                if a - 1 >= 0:
                    neighbour_idxs.append(a - 1)
                if a - 2 >= 0:
                    neighbour_idxs.append(a - 2)
                if b + 1 < nT:
                    neighbour_idxs.append(b + 1)
                if b + 2 < nT:
                    neighbour_idxs.append(b + 2)
                neighbour_idxs = sorted(set(neighbour_idxs))

                if not neighbour_idxs:
                    continue

                neighbour_vals = np.abs(vals_orig[neighbour_idxs])
                local_mean = neighbour_vals.mean()
                seg_vals = np.abs(vals_orig[a:b + 1])
                seg_max = seg_vals.max()

                if local_mean < eps:
                    ratio = math.inf if seg_max > 0 else 1.0
                else:
                    ratio = float(seg_max / (local_mean + eps))

                if seg_max > thresholds[col] and ratio >= ratio_thresh:
                    left_idx = a - 1 if a - 1 >= 0 else None
                    right_idx = b + 1 if b + 1 < nT else None
                    left_val = vals_orig[left_idx] if left_idx is not None else None
                    right_val = vals_orig[right_idx] if right_idx is not None else None

                    if (left_val is not None) and (right_val is not None):
                        length = right_idx - left_idx + 1
                        interp = np.linspace(left_val, right_val, length)
                        vals_new[a:b + 1] = interp[1:-1]
                    elif left_val is not None:
                        vals_new[a:b + 1] = left_val
                    elif right_val is not None:
                        vals_new[a:b + 1] = right_val
                    
                    total_segments += 1
                    total_cells += (b - a + 1)

            if not np.allclose(vals_new, vals_orig):
                df.loc[sub_idx_sorted, col] = vals_new

    if verbose:
        print(f"Outlier correction: {total_cells} cells in {total_segments} segments")
    
    return {"total_cells_corrected": total_cells, "total_segments": total_segments}

TIERA_QUANTILE = 0.999
TIERA_MULTIPLIER = 10.0
RATIO_THRESH = 1e4

global_thresholds = compute_global_thresholds(train_df, JOINT_COLS, q=TIERA_QUANTILE, multiplier=TIERA_MULTIPLIER)
train_summary = correct_glitches(train_df, JOINT_COLS, global_thresholds, id_col=ID_COL, time_col=TIME_COL,
                                 ratio_thresh=RATIO_THRESH, verbose=True, labels_df=train_labels_df,
                                 label_col="label", exclude_labels=['high_pain'])

Number of joint features: 29
Outlier correction: 7 cells in 4 segments


## Add Fourier Features

In [18]:
def add_fourier_features_df(df, time_col='time', periods=[4.0, 7.5]):
    for period in periods:
        freq = 2 * np.pi / period
        df[f'sin_p{period}'] = np.sin(freq * df[time_col]).astype(np.float32)
        df[f'cos_p{period}'] = np.cos(freq * df[time_col]).astype(np.float32)
    print(f"Added Fourier features for periods {periods}: {len(periods)*2} new features")
    return df

train_df = add_fourier_features_df(train_df, time_col=TIME_COL, periods=[4.0, 7.5])
test_df = add_fourier_features_df(test_df, time_col=TIME_COL, periods=[4.0, 7.5])

Added Fourier features for periods [4.0, 7.5]: 4 new features
Added Fourier features for periods [4.0, 7.5]: 4 new features


## Feature Scaling and Windowing

In [19]:
all_feature_cols = [c for c in train_df.columns if c not in [ID_COL, TIME_COL, TARGET_COL]]
test_feature_cols = [c for c in test_df.columns if c not in [ID_COL, TIME_COL]]
STATIC_COLS = [STATIC_COL]
dyn_feature_cols = [c for c in all_feature_cols if c not in STATIC_COLS]
dyn_feature_cols_test = [c for c in test_feature_cols if c not in STATIC_COLS]

print(f"Dynamic features: {len(dyn_feature_cols)}, Static features: {len(STATIC_COLS)}")

train_df = train_df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)
test_df = test_df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)

scaler = StandardScaler()
train_dyn_vals = scaler.fit_transform(train_df[dyn_feature_cols])
test_dyn_vals = scaler.transform(test_df[dyn_feature_cols])

train_df_scaled = train_df.copy()
test_df_scaled = test_df.copy()
train_df_scaled[dyn_feature_cols] = train_dyn_vals
test_df_scaled[dyn_feature_cols] = test_dyn_vals

label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(train_df_scaled[TARGET_COL].values)
print("Classes:", list(label_encoder.classes_))

def build_sequences(df_scaled, is_train=True):
    ids = df_scaled[ID_COL].unique()
    ids_sorted = np.sort(ids)
    seq_dyn = []
    seq_static = []
    seq_labels = []
    seq_ids = []

    for sid in ids_sorted:
        sub = df_scaled[df_scaled[ID_COL] == sid].sort_values(TIME_COL)
        dyn = sub[dyn_feature_cols].values
        static_vals = sub[STATIC_COLS].iloc[0].values.astype(float)
        seq_dyn.append(dyn)
        seq_static.append(static_vals)
        seq_ids.append(sid)
        if is_train:
            lbl = sub[TARGET_COL].iloc[0]
            seq_labels.append(lbl)

    seq_dyn = np.array(seq_dyn, dtype=object)
    seq_static = np.stack(seq_static)
    if is_train:
        seq_labels = np.array(label_encoder.transform(seq_labels))
        return seq_dyn, seq_static, seq_labels, np.array(seq_ids)
    else:
        return seq_dyn, seq_static, np.array(seq_ids)

train_dyn_seqs, train_static_seqs, y_seqs, train_ids = build_sequences(train_df_scaled, is_train=True)
test_dyn_seqs, test_static_seqs, test_ids = build_sequences(test_df_scaled, is_train=False)

def create_windows_from_sequences(dyn_seqs, static_seqs, labels, ids, window_size=30, stride=15, is_train=True):
    dyn_windows = []
    static_windows = []
    y_windows = []
    id_windows = []

    for dyn, static_vec, lbl, sid in zip(dyn_seqs, static_seqs,
                                         labels if is_train else [None]*len(dyn_seqs), ids):
        T = dyn.shape[0]
        if T < window_size:
            continue
        for start in range(0, T - window_size + 1, stride):
            end = start + window_size
            dyn_windows.append(dyn[start:end])
            static_windows.append(static_vec)
            id_windows.append(sid)
            if is_train:
                y_windows.append(lbl)

    dyn_windows = np.stack(dyn_windows).astype(np.float32)
    static_windows = np.stack(static_windows).astype(np.float32)
    id_windows = np.array(id_windows)
    if is_train:
        y_windows = np.array(y_windows)
        return dyn_windows, static_windows, y_windows, id_windows
    else:
        return dyn_windows, static_windows, id_windows

WINDOW_SIZE = 30
STRIDE = 15

X_dyn_win, X_static_win, y_win, ids_win = create_windows_from_sequences(
    train_dyn_seqs, train_static_seqs, y_seqs, train_ids,
    window_size=WINDOW_SIZE, stride=STRIDE, is_train=True)

X_dyn_test_win, X_static_test_win, test_ids_win = create_windows_from_sequences(
    test_dyn_seqs, test_static_seqs, labels=None, ids=test_ids,
    window_size=WINDOW_SIZE, stride=STRIDE, is_train=False)

print(f"Train windows: {X_dyn_win.shape}, Test windows: {X_dyn_test_win.shape}")

Dynamic features: 37, Static features: 1
Classes: ['high_pain', 'low_pain', 'no_pain']
Train windows: (5949, 30, 37), Test windows: (11916, 30, 37)


## Class Weights for Imbalanced Data

In [20]:
num_classes = len(np.unique(y_win))
class_counts = np.bincount(y_win, minlength=num_classes)
print("Class counts:", class_counts)

beta = 0.999
effective_num = 1.0 - np.power(beta, class_counts)
effective_num = np.maximum(effective_num, 1e-8)
weights = (1.0 - beta) / effective_num
weights = weights / weights.mean()

print("Class weights:", weights)
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

Class counts: [ 504  846 4599]
Class weights: [1.43294582 0.99379126 0.57326292]


## Dataset and DataLoader

In [21]:
class TimeSeriesWindowDataset(Dataset):
    def __init__(self, X_dyn, X_static, y=None):
        self.X_dyn = X_dyn
        self.X_static = X_static
        self.y = y

    def __len__(self):
        return len(self.X_dyn)

    def __getitem__(self, idx):
        dyn_np = np.asarray(self.X_dyn[idx], dtype=np.float32)
        stat_np = np.asarray(self.X_static[idx], dtype=np.float32)
        dyn = torch.from_numpy(dyn_np)
        stat = torch.from_numpy(stat_np)
        if self.y is not None:
            label = torch.tensor(self.y[idx], dtype=torch.long)
            return dyn, stat, label
        else:
            return dyn, stat, torch.tensor(-1, dtype=torch.long)

def make_loader(dataset, batch_size, shuffle):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)

## Feedforward Neural Network with Statistical Features

In [22]:
class SimpleFeedForwardNet(nn.Module):
    def __init__(self, input_size, static_dim, num_classes, hidden_sizes=[256, 128], dropout=0.3):
        super().__init__()
        
        # Extract statistical features from window
        # Per feature: mean, std, min, max, first, last, trend (last-first)
        self.num_features = input_size
        self.stats_per_feature = 7
        feature_size = self.num_features * self.stats_per_feature
        
        layers = []
        prev_size = feature_size + static_dim
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)
    
    def extract_statistical_features(self, x_dyn):
        # x_dyn: (B, window_size, features)
        # Compute statistics along time dimension
        mean = x_dyn.mean(dim=1)  # (B, features)
        std = x_dyn.std(dim=1)    # (B, features)
        min_val = x_dyn.min(dim=1)[0]  # (B, features)
        max_val = x_dyn.max(dim=1)[0]  # (B, features)
        first = x_dyn[:, 0, :]    # (B, features)
        last = x_dyn[:, -1, :]    # (B, features)
        trend = last - first      # (B, features)
        
        # Concatenate all statistics
        stats = torch.cat([mean, std, min_val, max_val, first, last, trend], dim=1)
        return stats  # (B, features * 7)
    
    def forward(self, x_dyn, x_static):
        # x_dyn: (B, window_size, features)
        # x_static: (B, static_dim)
        features = self.extract_statistical_features(x_dyn)
        x = torch.cat([features, x_static], dim=1)
        return self.network(x)

## Training Functions

In [23]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for xb_dyn, xb_static, yb in loader:
        xb_dyn = xb_dyn.to(device)
        xb_static = xb_static.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb_dyn, xb_static)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * yb.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

    avg_loss = total_loss / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, f1

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xb_dyn, xb_static, yb in loader:
            xb_dyn = xb_dyn.to(device)
            xb_static = xb_static.to(device)
            yb = yb.to(device)

            logits = model(xb_dyn, xb_static)
            loss = criterion(logits, yb)
            total_loss += loss.item() * yb.size(0)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    avg_loss = total_loss / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, f1

def train_model(model, train_loader, val_loader, criterion, learning_rate, epochs, patience, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    
    best_val_f1 = float("-inf")
    best_state_dict = None
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_f1 = validate(model, val_loader, criterion, device)
        scheduler.step(val_f1)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:03d} | train_loss={train_loss:.4f} f1={train_f1:.4f} | "
                  f"val_loss={val_loss:.4f} f1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state_dict = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch}")
                break

    return best_state_dict, best_val_f1

## K-Fold Cross Validation

In [24]:
K_FOLDS = 5
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 200
PATIENCE = 30
HIDDEN_SIZES = [512, 256, 128]
DROPOUT = 0.3

input_size = X_dyn_win.shape[2]  # num_features (we extract stats)
static_dim = X_static_win.shape[1]

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15)

sgkf = StratifiedGroupKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_f1_scores = []
model_paths = []

os.makedirs("models", exist_ok=True)

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_dyn_win, y_win, groups=ids_win), 1):
    print(f"\n{'='*60}")
    print(f"Fold {fold}/{K_FOLDS}")
    print(f"{'='*60}")
    
    X_tr_dyn, X_val_dyn = X_dyn_win[train_idx], X_dyn_win[val_idx]
    X_tr_stat, X_val_stat = X_static_win[train_idx], X_static_win[val_idx]
    y_tr, y_val = y_win[train_idx], y_win[val_idx]

    train_ds = TimeSeriesWindowDataset(X_tr_dyn, X_tr_stat, y_tr)
    val_ds = TimeSeriesWindowDataset(X_val_dyn, X_val_stat, y_val)

    train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = make_loader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

    model = SimpleFeedForwardNet(
        input_size=input_size,
        static_dim=static_dim,
        num_classes=num_classes,
        hidden_sizes=HIDDEN_SIZES,
        dropout=DROPOUT
    ).to(DEVICE)
    
    best_state_dict, best_val_f1 = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        patience=PATIENCE,
        device=DEVICE
    )

    fold_f1_scores.append(best_val_f1)
    
    fold_path = f"models/fold_{fold}_best.pt"
    torch.save(best_state_dict, fold_path)
    model_paths.append(fold_path)
    print(f"  Best validation F1: {best_val_f1:.4f}")

print(f"\n{'='*60}")
print("K-Fold Cross Validation Results")
print(f"{'='*60}")
print(f"Mean F1 Score: {np.mean(fold_f1_scores):.4f} ± {np.std(fold_f1_scores):.4f}")
print(f"Individual fold F1 scores: {[f'{f:.4f}' for f in fold_f1_scores]}")


Fold 1/5
  Epoch 001 | train_loss=0.9655 f1=0.7603 | val_loss=0.8722 f1=0.8582
  Epoch 010 | train_loss=0.6325 f1=0.9681 | val_loss=0.7283 f1=0.9204
  Epoch 020 | train_loss=0.5995 f1=0.9850 | val_loss=0.7016 f1=0.9376
  Epoch 030 | train_loss=0.5908 f1=0.9885 | val_loss=0.7135 f1=0.9327
  Epoch 040 | train_loss=0.5711 f1=0.9964 | val_loss=0.7056 f1=0.9356
  Epoch 050 | train_loss=0.5693 f1=0.9977 | val_loss=0.7154 f1=0.9368
  Epoch 060 | train_loss=0.5674 f1=0.9973 | val_loss=0.7150 f1=0.9349
  Epoch 070 | train_loss=0.5649 f1=0.9981 | val_loss=0.7011 f1=0.9421
  Early stopping at epoch 71
  Best validation F1: 0.9431

Fold 2/5
  Epoch 001 | train_loss=0.9522 f1=0.7695 | val_loss=0.8667 f1=0.8510
  Epoch 010 | train_loss=0.6363 f1=0.9692 | val_loss=0.7380 f1=0.9214
  Epoch 020 | train_loss=0.6010 f1=0.9836 | val_loss=0.7309 f1=0.9217
  Epoch 030 | train_loss=0.5856 f1=0.9927 | val_loss=0.7269 f1=0.9228
  Epoch 040 | train_loss=0.5778 f1=0.9946 | val_loss=0.7137 f1=0.9302
  Epoch 050 

## Attention-Based Window Aggregation with K-Fold

In [29]:
class AttentionAggregator(nn.Module):
    def __init__(self, num_classes, hidden_dim=32, dropout=0.3):
        super().__init__()
        self.query = nn.Linear(num_classes, hidden_dim)
        self.key = nn.Linear(num_classes, hidden_dim)
        self.value = nn.Linear(num_classes, num_classes)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, window_logits):
        # window_logits: (num_windows, num_classes)
        Q = self.query(window_logits)
        K = self.key(window_logits)
        V = self.value(window_logits)
        
        # Attention scores
        scores = torch.matmul(Q, K.T) / np.sqrt(Q.size(-1))
        attn_weights = F.softmax(scores.mean(dim=1), dim=0)
        attn_weights = self.dropout(attn_weights)
        
        # Weighted sum
        output = torch.sum(V * attn_weights.unsqueeze(-1), dim=0)
        return output

def train_attention_kfold(base_model_paths, X_dyn, X_static, y, ids, device, n_splits=5, epochs=30, lr=1e-3, 
                          input_size=None, static_dim=None, num_classes=None, hidden_sizes=None, dropout=0.3):
    """Train attention aggregators using K-fold with OOF predictions from base models."""
    print("\nTraining attention aggregators with K-fold cross-validation...")
    
    # Group data by sample_id
    unique_ids = np.unique(ids)
    sample_indices = {sid: np.where(ids == sid)[0] for sid in unique_ids}
    sample_labels = {sid: y[sample_indices[sid][0]] for sid in unique_ids}
    
    # K-fold split on samples (not windows)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    sample_id_array = unique_ids
    sample_label_array = np.array([sample_labels[sid] for sid in unique_ids])
    
    attention_models = []
    fold_val_f1 = []
    fold_val_f1_micro = []
    
    for fold_idx, (train_sample_idx, val_sample_idx) in enumerate(sgkf.split(sample_id_array, sample_label_array, groups=sample_id_array), 1):
        print(f"\n  Attention Fold {fold_idx}/{n_splits}")
        
        train_sample_ids = sample_id_array[train_sample_idx]
        val_sample_ids = sample_id_array[val_sample_idx]
        
        # Get OOF predictions from base models for this fold
        # Use the base model that was NOT trained on these samples
        base_model = SimpleFeedForwardNet(
            input_size=input_size,
            static_dim=static_dim,
            num_classes=num_classes,
            hidden_sizes=hidden_sizes,
            dropout=dropout
        ).to(device)
        
        # Load the base model trained on the opposite fold
        base_model_path = base_model_paths[fold_idx - 1]
        state_dict = torch.load(base_model_path, map_location=device)
        base_model.load_state_dict(state_dict)
        base_model.eval()
        
        # Get window predictions
        ds = TimeSeriesWindowDataset(X_dyn, X_static, y=None)
        loader = make_loader(ds, batch_size=128, shuffle=False)
        
        all_window_logits = []
        with torch.no_grad():
            for xb_dyn, xb_static, _ in loader:
                xb_dyn, xb_static = xb_dyn.to(device), xb_static.to(device)
                logits = base_model(xb_dyn, xb_static)
                all_window_logits.append(logits.cpu())
        
        all_window_logits = torch.cat(all_window_logits, dim=0)
        
        # Prepare training data for attention
        train_data = []
        train_labels = []
        for sid in train_sample_ids:
            mask = ids == sid
            sample_logits = all_window_logits[mask]
            train_data.append(sample_logits)
            train_labels.append(sample_labels[sid])
        
        # Prepare validation data
        val_data = []
        val_labels = []
        for sid in val_sample_ids:
            mask = ids == sid
            sample_logits = all_window_logits[mask]
            val_data.append(sample_logits)
            val_labels.append(sample_labels[sid])
        
        # Train attention model
        attention_model = AttentionAggregator(num_classes=num_classes, hidden_dim=32, dropout=dropout).to(device)
        optimizer = torch.optim.Adam(attention_model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()
        
        best_val_f1 = 0
        best_state = None
        patience_counter = 0
        
        for epoch in range(epochs):
            # Training
            attention_model.train()
            train_loss = 0
            for logits, label in zip(train_data, train_labels):
                logits = logits.to(device)
                label = torch.tensor(label, dtype=torch.long).to(device)
                
                optimizer.zero_grad()
                output = attention_model(logits)
                loss = criterion(output.unsqueeze(0), label.unsqueeze(0))
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
            
            # Validation
            attention_model.eval()
            val_preds = []
            with torch.no_grad():
                for logits in val_data:
                    logits = logits.to(device)
                    output = attention_model(logits)
                    val_preds.append(output.argmax().item())
            
            val_f1 = f1_score(val_labels, val_preds, average='weighted')
            val_f1_micro = f1_score(val_labels, val_preds, average='micro')
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = attention_model.state_dict()
                patience_counter = 0
            else:
                patience_counter += 1
            
            if (epoch + 1) % 10 == 0:
                print(f"    Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_data):.4f} | Val F1: {val_f1:.4f}")
            
            if patience_counter >= 10:
                print(f"    Early stopping at epoch {epoch+1}")
                break
        
        # Load best model
        attention_model.load_state_dict(best_state)
        attention_models.append(attention_model)
        fold_val_f1.append(best_val_f1)
        fold_val_f1_micro.append(val_f1_micro)
        print(f"    Best Val F1: {best_val_f1:.4f}")
    
    print(f"\n  Attention K-Fold Mean F1: {np.mean(fold_val_f1):.4f} ± {np.std(fold_val_f1):.4f}")
    print(f"  Attention K-Fold Micro F1: {np.mean(fold_val_f1_micro):.4f} ± {np.std(fold_val_f1_micro):.4f}")
    return attention_models

def predict_with_attention_ensemble(base_model_paths, attention_models, X_dyn, X_static, window_ids, device, batch_size=128,
                                   input_size=None, static_dim=None, num_classes=None, hidden_sizes=None, dropout=0.3):
    """Predict using ensemble of attention models."""
    unique_ids = np.unique(window_ids)
    
    # Collect window predictions from all base models
    all_window_logits = []
    for path in base_model_paths:
        model = SimpleFeedForwardNet(
            input_size=input_size,
            static_dim=static_dim,
            num_classes=num_classes,
            hidden_sizes=hidden_sizes,
            dropout=dropout
        ).to(device)
        state_dict = torch.load(path, map_location=device)
        model.load_state_dict(state_dict)
        model.eval()
        
        window_logits = []
        ds = TimeSeriesWindowDataset(X_dyn, X_static, y=None)
        loader = make_loader(ds, batch_size=batch_size, shuffle=False)
        
        with torch.no_grad():
            for xb_dyn, xb_static, _ in loader:
                xb_dyn, xb_static = xb_dyn.to(device), xb_static.to(device)
                logits = model(xb_dyn, xb_static)
                window_logits.append(logits.cpu())
        
        all_window_logits.append(torch.cat(window_logits, dim=0))
    
    # Average across base model folds
    mean_window_logits = torch.stack(all_window_logits).mean(dim=0).to(device)
    
    # Aggregate per sample using ensemble of attention models
    all_attention_preds = []
    for attention_model in attention_models:
        attention_model.eval()
        fold_preds = []
        with torch.no_grad():
            for sid in unique_ids:
                mask = window_ids == sid
                sample_logits = mean_window_logits[mask]
                output = attention_model(sample_logits)
                fold_preds.append(output.cpu().numpy())
        all_attention_preds.append(np.array(fold_preds))
    
    # Average predictions across attention folds
    mean_probs = np.mean(all_attention_preds, axis=0)
    predictions = mean_probs.argmax(axis=1)
    
    return unique_ids, predictions

# Train attention aggregators with K-fold
attention_models = train_attention_kfold(
    base_model_paths=model_paths,
    X_dyn=X_dyn_win,
    X_static=X_static_win,
    y=y_win,
    ids=ids_win,
    device=DEVICE,
    n_splits=K_FOLDS,
    epochs=30,
    lr=1e-3,
    input_size=input_size,
    static_dim=static_dim,
    num_classes=num_classes,
    hidden_sizes=HIDDEN_SIZES,
    dropout=DROPOUT
)

print("\nGenerating predictions using attention ensemble...")
sample_ids_ordered, preds = predict_with_attention_ensemble(
    base_model_paths=model_paths,
    attention_models=attention_models,
    X_dyn=X_dyn_test_win,
    X_static=X_static_test_win,
    window_ids=test_ids_win,
    device=DEVICE,
    batch_size=BATCH_SIZE * 2,
    input_size=input_size,
    static_dim=static_dim,
    num_classes=num_classes,
    hidden_sizes=HIDDEN_SIZES,
    dropout=DROPOUT
)

preds_labels = label_encoder.inverse_transform(preds)


Training attention aggregators with K-fold cross-validation...

  Attention Fold 1/5
    Epoch 10/30 | Train Loss: 0.0124 | Val F1: 0.9446
    Early stopping at epoch 13
    Best Val F1: 0.9537

  Attention Fold 2/5
    Epoch 10/30 | Train Loss: 0.0081 | Val F1: 0.9351
    Early stopping at epoch 11
    Best Val F1: 0.9440

  Attention Fold 3/5
    Epoch 10/30 | Train Loss: 0.0069 | Val F1: 0.9394
    Early stopping at epoch 11
    Best Val F1: 0.9394

  Attention Fold 4/5
    Epoch 10/30 | Train Loss: 0.0129 | Val F1: 0.9519
    Early stopping at epoch 17
    Best Val F1: 0.9519

  Attention Fold 5/5
    Epoch 10/30 | Train Loss: 0.0128 | Val F1: 0.9220
    Early stopping at epoch 11
    Best Val F1: 0.9441

  Attention K-Fold Mean F1: 0.9466 ± 0.0054
  Attention K-Fold Micro F1: 0.9410 ± 0.0101

Generating predictions using attention ensemble...


## Create Submission File

In [28]:
submission = pd.DataFrame({
    ID_COL: sample_ids_ordered,
    TARGET_COL: preds_labels
})
submission = submission.sort_values(ID_COL).reset_index(drop=True)
submission.to_csv("submission_simple_ffnn.csv", index=False)

print("\n" + "="*60)
print("SUBMISSION FILE CREATED!")
print("="*60)
print(submission.head(10))
print(f"\nTotal samples: {len(submission)}")
print(f"\nClass distribution:")
print(submission[TARGET_COL].value_counts())


SUBMISSION FILE CREATED!
   sample_index    label
0             0  no_pain
1             1  no_pain
2             2  no_pain
3             3  no_pain
4             4  no_pain
5             5  no_pain
6             6  no_pain
7             7  no_pain
8             8  no_pain
9             9  no_pain

Total samples: 1324

Class distribution:
label
no_pain      1058
low_pain      175
high_pain      91
Name: count, dtype: int64
